In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (SparkSession.builder
    .appName("Instacart-Day1")
    .config("spark.sql.shuffle.partitions", "8")  # local: ลดจาก default 200
    .config("spark.driver.memory", "4g")           # สำคัญ — order_products ใหญ่
    .getOrCreate())

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark version: 3.5.0
Spark UI: http://a41fb29cf9be:4040


In [2]:
DATA_PATH = "/home/jovyan/work/data/raw"

aisles = spark.read.csv(f"{DATA_PATH}/aisles.csv", header=True, inferSchema=True)
departments = spark.read.csv(f"{DATA_PATH}/departments.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA_PATH}/products.csv", header=True, inferSchema=True)

aisles.show(5)
departments.show()
products.printSchema()
print("Products count:", products.count())

+--------+--------------------+
|aisle_id|               aisle|
+--------+--------------------+
|       1|prepared soups sa...|
|       2|   specialty cheeses|
|       3| energy granola bars|
|       4|       instant foods|
|       5|marinades meat pr...|
+--------+--------------------+
only showing top 5 rows

+-------------+---------------+
|department_id|     department|
+-------------+---------------+
|            1|         frozen|
|            2|          other|
|            3|         bakery|
|            4|        produce|
|            5|        alcohol|
|            6|  international|
|            7|      beverages|
|            8|           pets|
|            9|dry goods pasta|
|           10|           bulk|
|           11|  personal care|
|           12|   meat seafood|
|           13|         pantry|
|           14|      breakfast|
|           15|   canned goods|
|           16|     dairy eggs|
|           17|      household|
|           18|         babies|
|           19|

In [3]:
# Define schema explicitly — เร็วกว่า inferSchema และเป็น best practice ใน production
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("user_id", IntegerType(), False),
    StructField("eval_set", StringType(), True),
    StructField("order_number", IntegerType(), True),
    StructField("order_dow", IntegerType(), True),
    StructField("order_hour_of_day", IntegerType(), True),
    StructField("days_since_prior_order", DoubleType(), True),  # มี NULL สำหรับ first order
])

orders = spark.read.csv(f"{DATA_PATH}/orders.csv", header=True, schema=orders_schema)

order_products_prior = spark.read.csv(
    f"{DATA_PATH}/order_products__prior.csv",
    header=True, inferSchema=True
)

orders.printSchema()
print("Orders count:", orders.count())          # ~3.4M
print("Order items count:", order_products_prior.count())  # ~32M

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)

Orders count: 3421083
Order items count: 32434489


In [4]:
BRONZE = "/home/jovyan/work/data/bronze"

(orders.write
   .mode("overwrite")
   .parquet(f"{BRONZE}/orders"))

(order_products_prior.write
   .mode("overwrite")
   .parquet(f"{BRONZE}/order_products_prior"))

(products.write.mode("overwrite").parquet(f"{BRONZE}/products"))
(aisles.write.mode("overwrite").parquet(f"{BRONZE}/aisles"))
(departments.write.mode("overwrite").parquet(f"{BRONZE}/departments"))